# Phase 2 — BGP Time-Series Data Capture

This notebook is part of the **BGP AI Analytics Engine** project.

The purpose of this notebook is to capture live BGP updates from the **Route Views BMP streaming service** using `pybgpstream` and save the received BGP events into a persistent dataset for further analysis.

## Objective

Establish a live BGP streaming connection and transform incoming BGP update elements into structured time-series records.

The captured data will include:

- Timestamp
- Collector
- Router
- Router IP
- Peer ASN
- Peer address
- BGP update type (Announcement / Withdrawal)
- Prefix
- Next-hop
- AS Path

## Data Flow

Route Views BMP Stream

↓

`pybgpstream`

↓

BGP Update Elements

↓

Structured Time-Series Records

↓

CSV Dataset

↓

Future BGP Time-Series Analysis

## Capture Strategy

The streaming data will be captured for a defined period of time and written to a CSV file under:

`data/raw_data/`

The raw streaming dataset will be preserved for subsequent processing, profiling, and analysis.

## Phase 2 Direction

Phase 2 extends the project from **static BGP dataset analysis** toward **dynamic, time-series BGP analytics**.

The captured data will eventually be used to investigate:

- BGP update rates
- Announcement and withdrawal patterns
- Prefix changes over time
- AS-path changes
- Peer behavior
- Route instability
- BGP events and anomalies

> **Note:** This notebook focuses on data acquisition and persistence. Analysis and AI/ML processing will be performed in subsequent stages.

In [1]:
import pybgpstream
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

# ============================================================
# BGP TIME-SERIES CAPTURE
# ============================================================

CAPTURE_DURATION = 10 * 60   # 10 minutes

OUTPUT_DIR = Path("../data/raw_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

start_datetime = datetime.now(timezone.utc)
start_time = time.time()

filename = (
    f"bgp_updates_"
    f"{start_datetime.strftime('%Y%m%d_%H%M%S')}_UTC.csv"
)

output_file = OUTPUT_DIR / filename

print("==============================================")
print("BGP TIME-SERIES CAPTURE")
print("==============================================")
print(f"Start time : {start_datetime}")
print(f"Duration   : {CAPTURE_DURATION / 60:.0f} minutes")
print(f"Output     : {output_file}")
print()

# ------------------------------------------------------------
# Connect to Route Views BMP stream
# ------------------------------------------------------------

stream = pybgpstream.BGPStream(
    project="routeviews-stream"
)

print("Connected to Route Views BMP stream")
print("Capturing BGP updates...")
print()

# ------------------------------------------------------------
# CSV output
# ------------------------------------------------------------

fieldnames = [
    "timestamp",
    "collector",
    "router",
    "router_ip",
    "peer_asn",
    "peer_address",
    "type",
    "prefix",
    "next_hop",
    "as_path",
]

count = 0
announcements = 0
withdrawals = 0

with open(output_file, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=fieldnames
    )

    writer.writeheader()

    # --------------------------------------------------------
    # Read live BGP stream
    # --------------------------------------------------------

    for elem in stream:

        elapsed = time.time() - start_time

        # Stop after configured capture duration
        if elapsed >= CAPTURE_DURATION:
            break

        # Only capture BGP announcements and withdrawals
        if elem.type not in ("A", "W"):
            continue

        # Record timestamp
        timestamp = datetime.fromtimestamp(
            elem.time,
            tz=timezone.utc
        ).isoformat()

        # Record-specific information
        record = elem.record

        # Fields common to announcements / withdrawals
        prefix = elem.fields.get("prefix")

        # These fields are available for announcements,
        # but not for withdrawals.
        next_hop = elem.fields.get("next-hop")
        as_path = elem.fields.get("as-path")

        row = {
            "timestamp": timestamp,
            "collector": record.collector,
            "router": record.router,
            "router_ip": getattr(record, "router_ip", None),
            "peer_asn": elem.peer_asn,
            "peer_address": elem.peer_address,
            "type": elem.type,
            "prefix": prefix,
            "next_hop": next_hop,
            "as_path": as_path,
        }

        writer.writerow(row)

        count += 1

        if elem.type == "A":
            announcements += 1
        elif elem.type == "W":
            withdrawals += 1

        # Flush periodically so data is physically written
        # during the long-running capture.
        if count % 10000 == 0:
            csvfile.flush()

        # Progress display
        if count % 100000 == 0:
            print(
                f"[{elapsed:6.0f}s] "
                f"Updates: {count:,} | "
                f"Announcements: {announcements:,} | "
                f"Withdrawals: {withdrawals:,}"
            )

# ------------------------------------------------------------
# Capture summary
# ------------------------------------------------------------

end_datetime = datetime.now(timezone.utc)
total_time = time.time() - start_time

print()
print("==============================================")
print("BGP TIME-SERIES CAPTURE COMPLETED")
print("==============================================")
print(f"Start time       : {start_datetime}")
print(f"End time         : {end_datetime}")
print(f"Duration         : {total_time:.1f} seconds")
print(f"Total updates    : {count:,}")
print(f"Announcements    : {announcements:,}")
print(f"Withdrawals      : {withdrawals:,}")
print(f"Average rate     : {count / total_time:.2f} elements/sec")
print(f"Output file      : {output_file}")
print("==============================================")

BGP TIME-SERIES CAPTURE
Start time : 2026-08-20 01:48:57.541694+00:00
Duration   : 10 minutes
Output     : ../data/raw_data/bgp_updates_20260820_014857_UTC.csv

Connected to Route Views BMP stream
Capturing BGP updates...

[    34s] Updates: 100,000 | Announcements: 90,488 | Withdrawals: 9,512
[    50s] Updates: 200,000 | Announcements: 181,008 | Withdrawals: 18,992
[    69s] Updates: 300,000 | Announcements: 273,488 | Withdrawals: 26,512
[    88s] Updates: 400,000 | Announcements: 366,074 | Withdrawals: 33,926
[   103s] Updates: 500,000 | Announcements: 459,773 | Withdrawals: 40,227
[   121s] Updates: 600,000 | Announcements: 550,911 | Withdrawals: 49,089
[   139s] Updates: 700,000 | Announcements: 641,718 | Withdrawals: 58,282
[   171s] Updates: 800,000 | Announcements: 733,205 | Withdrawals: 66,795
[   179s] Updates: 900,000 | Announcements: 823,197 | Withdrawals: 76,803
[   204s] Updates: 1,000,000 | Announcements: 914,392 | Withdrawals: 85,608
[   221s] Updates: 1,100,000 | Announ

## Capture Result

The live BGP streaming capture was successfully completed.

The 10-minute capture produced more than 3.1 million BGP update elements, including both announcements and withdrawals.

The resulting raw time-series dataset has been saved under `data/raw_data/` for subsequent processing and analysis.

This dataset represents the first dynamic BGP dataset in the project and will be used in the next stages for BGP time-series analysis, route-change analysis, and eventually anomaly detection and AI/ML applications.